In [ ]:
import pandas as pd

fixtures = pd.read_csv("../data/raw/group_fixtures.csv")
print(fixtures.shape)
fixtures.head()

(72, 6)


,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [ ]:
knockout = pd.read_csv("../data/raw/knockout_slots.csv")
print(knockout.shape)
knockout.head()

(32, 7)


,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I


In [ ]:
elo_ratings = {
    "Brazil": 2000,
    "France": 1980,
    "England": 1850,
    "Germany": 1900,
    "Mexico": 1700,
    "Saudi Arabia": 1500,
}
print(elo_ratings['France'])

1980


In [ ]:
def update_elo(rating_a, rating_b, result_a, k=30):
    expected = 1 / (1 + 10 ** ((rating_b - rating_a) / 400))
    new_a = rating_a + k * (result_a - expected)
    new_b = rating_b + k * ((1 - result_a) - (1 - expected))
    return round(new_a, 1), round(new_b, 1)

In [ ]:
new_brazil, new_vatican = update_elo(2000, 800, result_a=0)
print(f"Brazil: 2000 → {new_brazil}")
print(f"Vatican City: 800 → {new_vatican}")

Brazil: 2000 → 1970.0
Vatican City: 800 → 830.0


In [ ]:
home_teams = fixtures["home_team"].unique()
away_teams = fixtures["away_team"].unique()
all_teams = set(home_teams) | set(away_teams)
print(sorted(all_teams))

['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Cabo Verde', 'Canada', 'Colombia', 'Croatia', 'Curaçao', "Côte d'Ivoire", 'Ecuador', 'Egypt', 'England', 'FIFA Playoff 1', 'FIFA Playoff 2', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Switzerland', 'Tunisia', 'UEFA Playoff A', 'UEFA Playoff B', 'UEFA Playoff C', 'UEFA Playoff D', 'USA', 'Uruguay', 'Uzbekistan']


In [ ]:
results = pd.read_csv("../data/raw/results.csv")
print(results.shape)
results.head()

(49477, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [ ]:
results["date"] = pd.to_datetime(results["date"])
recent = results[results["date"] >= "2014-01-01"].copy()
print(recent.shape)

(11919, 9)


In [ ]:
elo = {}
for _, row in recent.sort_values("date").iterrows():
    home, away = row["home_team"], row["away_team"]
    if pd.isna(row["home_score"]) or pd.isna(row["away_score"]):
        continue
    r_home, r_away = elo.get(home, 1500), elo.get(away, 1500)
    if row["home_score"] > row["away_score"]:    result = 1.0
    elif row["home_score"] == row["away_score"]: result = 0.5
    else:                                         result = 0.0
    elo[home], elo[away] = update_elo(r_home, r_away, result)

In [ ]:
print(f"Teams rated: {len(elo)}")
print(f"Brazil: {elo.get('Brazil', 'N/A')}")
print(f"England: {elo.get('England', 'N/A')}")
print(f"Curaçao: {elo.get('Curaçao', 'N/A')}")

Teams rated: 300
Brazil: 1831.6
England: 1827.2
Curaçao: 1532.5


In [ ]:
elo_ratings = {team: elo.get(team, 1500) for team in all_teams}
print(f"Tournament teams rated: {len(elo_ratings)}")
print(sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)[:10])

Tournament teams rated: 48
[('Argentina', 1927.1), ('Spain', 1911.4), ('France', 1876.5), ('Morocco', 1873.7), ('Japan', 1843.7), ('Portugal', 1836.1), ('Germany', 1833.7), ('Brazil', 1831.6), ('England', 1827.2), ('Colombia', 1825.0)]


In [ ]:
def get_last_matches(team, n = 10):
    mask = (recent["home_team"] == team) | (recent["away_team"] == team)
    return recent[mask].sort_values("date").tail(n)

brazil_form = get_last_matches("Brazil")
print(brazil_form[["date", "home_team", "away_team", "home_score", "away_score"]])

            date home_team away_team  home_score  away_score
48811 2025-10-14     Japan    Brazil         3.0         2.0
48917 2025-11-15    Brazil   Senegal         2.0         0.0
48985 2025-11-18    Brazil   Tunisia         1.0         1.0
49122 2026-03-26    Brazil    France         1.0         2.0
49220 2026-03-31    Brazil   Croatia         3.0         1.0
49276 2026-05-31    Brazil    Panama         6.0         2.0
49347 2026-06-06    Brazil     Egypt         2.0         1.0
49410 2026-06-13    Brazil   Morocco         1.0         1.0
49434 2026-06-19    Brazil     Haiti         3.0         0.0
49457 2026-06-24  Scotland    Brazil         NaN         NaN


In [ ]:
def get_form_score(team, n=10):
    matches = get_last_matches(team, n)
    matches = matches.dropna(subset=["home_score", "away_score"])
    points = []
    for _, row in matches.iterrows():
        if row["home_team"] == team:
            pts = 1.0 if row["home_score"] > row["away_score"] else (0.5 if row["home_score"] == row["away_score"] else 0.0)
        else:
            pts = 1.0 if row["away_score"] > row["home_score"] else (0.5 if row["away_score"] == row["home_score"] else 0.0)
        points.append(pts)
    return sum(points) / len(points) if points else 0.5


In [ ]:
print(f"Brazil form:    {get_form_score('Brazil'):.3f}")
print(f"England form:   {get_form_score('England'):.3f}")
print(f"Curaçao form:   {get_form_score('Curaçao'):.3f}")


Brazil form:    0.667
England form:   0.778
Curaçao form:   0.389


In [ ]:
adjusted_elo = {
    team: elo_ratings[team] + (get_form_score(team) - 0.5) * 100
    for team in elo_ratings
}

print(f"England:  {elo_ratings['England']:.1f} → {adjusted_elo['England']:.1f}")
print(f"Curaçao:  {elo_ratings['Curaçao']:.1f} → {adjusted_elo['Curaçao']:.1f}")
print(f"Brazil:   {elo_ratings['Brazil']:.1f} → {adjusted_elo['Brazil']:.1f}")


England:  1827.2 → 1855.0
Curaçao:  1532.5 → 1521.4
Brazil:   1831.6 → 1848.3


In [ ]:
def win_probability(elo_a, elo_b):
    diff = elo_a - elo_b
    return 1 / (1 + 10 ** (-diff / 400))

prob = win_probability(adjusted_elo["England"], adjusted_elo["Curaçao"])
print(f"England win probability vs Curaçao: {prob:.3f}")


England win probability vs Curaçao: 0.872


In [ ]:
def expected_goals(elo_a, elo_b, base=1.3):
    p = win_probability(elo_a, elo_b)
    xg_a = base * (2 * p)
    xg_b = base * (2 * (1 - p))
    return round(xg_a, 2), round(xg_b, 2)

eng, cur = expected_goals(adjusted_elo["England"], adjusted_elo["Curaçao"])
print(f"England xG: {eng}")
print(f"Curaçao xG: {cur}")


England xG: 2.27
Curaçao xG: 0.33


## Stage 6 - most likely score vs optimal EV score

In [ ]:
EV = (0.14 * 25) + (0.73 * 10) + (0.13 * 0)
print(EV)

10.8


In [ ]:
from scipy.stats import poisson

def score_probabilities(xg_a, xg_b, max_goals=8):
    probs = {}
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            probs[(i, j)] = poisson.pmf(i, xg_a) * poisson.pmf(j, xg_b)
    return probs

probs = score_probabilities(2.27, 0.33)
print(f"P(2-0): {probs[(2,0)]:.3f}")
print(f"P(1-0): {probs[(1,0)]:.3f}")
print(f"P(0-0): {probs[(0,0)]:.3f}")


P(2-0): 0.191
P(1-0): 0.169
P(0-0): 0.074


In [ ]:
def best_prediction(xg_a, xg_b, points_exact=25, points_result=10, max_goals=8):
    probs = score_probabilities(xg_a, xg_b)
    p_home_win = sum(p for (i,j), p in probs.items() if i > j)
    p_draw     = sum(p for (i,j), p in probs.items() if i == j)
    p_away_win = sum(p for (i,j), p in probs.items() if i < j)

    best_ev, best_score = -1, (0, 0)
    for (i, j), p_exact in probs.items():
        if i > j:   p_result = p_home_win
        elif i == j: p_result = p_draw
        else:        p_result = p_away_win
        ev = p_exact * points_exact + (p_result - p_exact) * points_result
        if ev > best_ev:
            best_ev, best_score = ev, (i, j)
    return best_score, round(best_ev, 3)


In [ ]:
score, ev = best_prediction(2.27, 0.33)
print(f"Optimal prediction: {score[0]}-{score[1]}")
print(f"Expected value: {ev} points")
print(f"Most likely score: 2-0 (prob: {probs[(2,0)]:.3f})")

Optimal prediction: 2-0
Expected value: 11.046 points
Most likely score: 2-0 (prob: 0.191)


In [ ]:
xg_fra, xg_ger = expected_goals(adjusted_elo["France"], adjusted_elo["Germany"])
print(f"France xG: {xg_fra}, Germany xG: {xg_ger}")

score, ev = best_prediction(xg_fra, xg_ger)
probs_fg = score_probabilities(xg_fra, xg_ger)
most_likely = max(probs_fg, key=probs_fg.get)

print(f"Most likely score: {most_likely[0]}-{most_likely[1]}")
print(f"Optimal EV score: {score[0]}-{score[1]}")
print(f"Are they the same? {most_likely == score}")


France xG: 1.4, Germany xG: 1.2
Most likely score: 1-1
Optimal EV score: 1-0
Are they the same? False


## Stage 7 — Win / Draw / Loss Probabilities


In [ ]:
def match_probabilities(xg_a, xg_b):
    probs = score_probabilities(xg_a, xg_b)
    p_home = sum(p for (i,j), p in probs.items() if i > j)
    p_draw  = sum(p for (i,j), p in probs.items() if i == j)
    p_away  = sum(p for (i,j), p in probs.items() if i < j)
    return round(p_home, 3), round(p_draw, 3), round(p_away, 3)

hw, d, aw = match_probabilities(2.27, 0.33)
print(f"England win: {hw}, Draw: {d}, Curaçao win: {aw}")
print(f"Total: {hw + d + aw}")


England win: 0.818, Draw: 0.141, Curaçao win: 0.041
Total: 1.0


In [ ]:
h, d, a = match_probabilities(1.3, 1.3)
print(f"Home: {h}, Draw: {d}, Away: {a}")

Home: 0.368, Draw: 0.264, Away: 0.368


In [ ]:
style = {
    "Brazil":   {"corners": 1.2, "yellows": 1.0, "reds": 0.8},
    "England":  {"corners": 1.1, "yellows": 0.9, "reds": 0.7},
    "Curaçao":  {"corners": 0.8, "yellows": 1.1, "reds": 1.0},
}

def predict_corners_cards(team_a, team_b, base_corners=5, base_yellows=1.5, base_reds=0.1):
    c = base_corners * (style[team_a]["corners"] + style[team_b]["corners"])
    y = base_yellows * (style[team_a]["yellows"] + style[team_b]["yellows"])
    r = base_reds   * (style[team_a]["reds"]    + style[team_b]["reds"])
    return round(c, 1), round(y, 1), round(r, 1)

corners, yellows, reds = predict_corners_cards("England", "Curaçao")
print(f"Corners: {corners}, Yellows: {yellows}, Reds: {reds}")


Corners: 9.5, Yellows: 3.0, Reds: 0.2


In [ ]:
style = {
    "Argentina": {"corners": 1.1, "yellows": 1.2, "reds": 1.1},
    "Australia": {"corners": 0.9, "yellows": 1.0, "reds": 0.8},
    "Austria":   {"corners": 1.0, "yellows": 1.1, "reds": 0.9},
    "Belgium":   {"corners": 1.1, "yellows": 0.9, "reds": 0.7},
    "Brazil":    {"corners": 1.2, "yellows": 1.0, "reds": 0.8},
    "Cabo Verde":{"corners": 0.8, "yellows": 1.2, "reds": 1.1},
    "Canada":    {"corners": 1.0, "yellows": 1.0, "reds": 0.9},
    "Colombia":  {"corners": 1.1, "yellows": 1.3, "reds": 1.2},
    "Croatia":   {"corners": 1.0, "yellows": 1.0, "reds": 0.8},
    "Curaçao":   {"corners": 0.8, "yellows": 1.1, "reds": 1.0},
    "Côte d'Ivoire":{"corners": 1.0,"yellows": 1.1,"reds": 1.0},
    "Ecuador":   {"corners": 0.9, "yellows": 1.1, "reds": 1.0},
    "Egypt":     {"corners": 0.9, "yellows": 1.1, "reds": 1.0},
    "England":   {"corners": 1.1, "yellows": 0.9, "reds": 0.7},
    "France":    {"corners": 1.1, "yellows": 1.0, "reds": 0.8},
    "Germany":   {"corners": 1.0, "yellows": 0.9, "reds": 0.7},
    "Ghana":     {"corners": 0.9, "yellows": 1.1, "reds": 1.0},
    "Haiti":     {"corners": 0.8, "yellows": 1.2, "reds": 1.1},
    "Iran":      {"corners": 0.9, "yellows": 1.2, "reds": 1.0},
    "Japan":     {"corners": 1.0, "yellows": 0.8, "reds": 0.6},
    "Jordan":    {"corners": 0.8, "yellows": 1.1, "reds": 1.0},
    "Mexico":    {"corners": 1.0, "yellows": 1.1, "reds": 1.0},
    "Morocco":   {"corners": 0.9, "yellows": 1.1, "reds": 0.9},
    "Netherlands":{"corners":1.1, "yellows": 1.0, "reds": 0.8},
    "New Zealand":{"corners":0.8, "yellows": 0.9, "reds": 0.8},
    "Norway":    {"corners": 1.0, "yellows": 1.0, "reds": 0.8},
    "Panama":    {"corners": 0.8, "yellows": 1.2, "reds": 1.1},
    "Paraguay":  {"corners": 0.9, "yellows": 1.2, "reds": 1.1},
    "Portugal":  {"corners": 1.1, "yellows": 1.1, "reds": 0.9},
    "Qatar":     {"corners": 0.9, "yellows": 1.0, "reds": 0.9},
    "Algeria":   {"corners": 0.9, "yellows": 1.1, "reds": 1.0},
    "Saudi Arabia":{"corners":0.9,"yellows": 1.0, "reds": 0.9},
    "Scotland":  {"corners": 1.0, "yellows": 1.0, "reds": 0.8},
    "Senegal":   {"corners": 0.9, "yellows": 1.1, "reds": 1.0},
    "South Africa":{"corners":0.9,"yellows": 1.1, "reds": 1.0},
    "South Korea":{"corners":1.0, "yellows": 1.0, "reds": 0.8},
    "Spain":     {"corners": 1.2, "yellows": 0.9, "reds": 0.7},
    "Switzerland":{"corners":1.0, "yellows": 0.9, "reds": 0.7},
    "Tunisia":   {"corners": 0.9, "yellows": 1.2, "reds": 1.0},
    "Uruguay":   {"corners": 1.0, "yellows": 1.3, "reds": 1.2},
    "USA":       {"corners": 1.0, "yellows": 0.9, "reds": 0.8},
    "Uzbekistan":{"corners": 0.9, "yellows": 1.1, "reds": 1.0},
    "UEFA Playoff A":{"corners":1.0,"yellows":1.0,"reds":1.0},
    "UEFA Playoff B":{"corners":1.0,"yellows":1.0,"reds":1.0},
    "UEFA Playoff C":{"corners":1.0,"yellows":1.0,"reds":1.0},
    "UEFA Playoff D":{"corners":1.0,"yellows":1.0,"reds":1.0},
    "FIFA Playoff 1":{"corners":1.0,"yellows":1.0,"reds":1.0},
    "FIFA Playoff 2":{"corners":1.0,"yellows":1.0,"reds":1.0},
}


In [ ]:
corners, yellows, reds = predict_corners_cards("Brazil", "UEFA Playoff A")
print(f"Corners: {corners}, Yellows: {yellows}, Reds: {reds}")


Corners: 11.0, Yellows: 3.0, Reds: 0.2


In [ ]:
def predict_match(row):
    home, away = row["home_team"], row["away_team"]
    xg_h, xg_a = expected_goals(adjusted_elo.get(home, 1550), adjusted_elo.get(away, 1550))
    score, _ = best_prediction(xg_h, xg_a)
    hw, d, aw = match_probabilities(xg_h, xg_a)
    c, y, r = predict_corners_cards(home, away)
    return {"match_id": row["match_id"], "home_team": home, "away_team": away,
            "home_score": score[0], "away_score": score[1],
            "home_win%": hw, "draw%": d, "away_win%": aw,
            "corners": c, "yellows": y, "reds": r}


In [ ]:
group_predictions = pd.DataFrame([predict_match(row) for _, row in fixtures.iterrows()])
print(group_predictions.shape)
group_predictions.head()


(72, 11)


,match_id,home_team,away_team,home_score,away_score,home_win%,draw%,away_win%,corners,yellows,reds
0,1,Mexico,South Africa,2,0,0.736,0.181,0.083,9.5,3.3,0.2
1,2,South Korea,UEFA Playoff D,2,0,0.774,0.164,0.062,10.0,3.0,0.2
2,3,Canada,UEFA Playoff A,2,0,0.757,0.171,0.071,10.0,3.0,0.2
3,4,USA,Paraguay,0,1,0.130,0.211,0.659,9.5,3.2,0.2
4,5,Australia,UEFA Playoff C,2,0,0.727,0.185,0.088,9.5,3.0,0.2


In [ ]:
group_predictions.to_csv("../outputs/group_predictions.csv", index=False)
print("Saved.")

Saved.
